In [2]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from category_encoders import TargetEncoder
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.impute import SimpleImputer


In [27]:
%store -r x_train
x_train = x_train

%store -r y_train
y_train = y_train

%store -r x_val
x_val = x_val

%store -r y_val
y_val = y_val

%store -r x_test
x_test = x_test

%store -r y_test
y_test = y_test

In [4]:
# 1. UTFØR FREKVENSFILTRERING HER OGSÅ (slik at model_clean finnes)
min_frequency = 5
model_counts = x_train['model'].value_counts()
frequent_models = model_counts[model_counts >= min_frequency].index

x_train['model_clean'] = x_train['model'].apply(lambda x: x if x in frequent_models else 'other')
x_val['model_clean'] = x_val['model'].apply(lambda x: x if x in frequent_models else 'other')

# 2. Definer featurmene riktig med model_clean i stedet for model
numeric_features = ['year', 'odometer']
high_cardinality_features = ['manufacturer', 'model_clean', 'region']

low_cardinality_features = [
    'condition', 'cylinders', 'fuel', 'title_status', 
    'transmission', 'drive', 'type'
]

# 3. Felles preprocessor
preprocessor = ColumnTransformer([
    ('numeric', SimpleImputer(strategy='median'), numeric_features),
    ('target_encode', TargetEncoder(handle_missing='value', handle_unknown='value'), high_cardinality_features),
    ('one_hot', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), low_cardinality_features)
], remainder='drop')

## For hver modell, lage en spesifikk pipeline og bygger og trener modellen

benytter cross-validation

In [5]:
# Evalueringsfunksjon
def evaluate_model(model, x_train, y_train, x_val, y_val):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    cross_val_r2 = cross_val_score(model, x_train, y_train, cv=kf, scoring='r2')
    cross_val_rmse = -cross_val_score(model, x_train, y_train, cv=kf, scoring='neg_root_mean_squared_error')

    print(f"Cross-Validation R2: {cross_val_r2.mean():.4f}, Training CV RMSE: {cross_val_rmse.mean():.4f}")

    model.fit(x_train, y_train)

    y_train_pred = model.predict(x_train)
    y_val_pred = model.predict(x_val)

    train_r2 = r2_score(y_train, y_train_pred)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
    val_r2 = r2_score(y_val, y_val_pred)
    rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))
    
    print(f"Training R2: {train_r2:.4f}, Training RMSE: {rmse_train:.4f}")
    print(f"Validation R2: {val_r2:.4f}, Validation RMSE: {rmse_val:.4f}\n")

# Model Training

## Linear Regression

Sjekker antall NaN som oppstår og handtere dem med simpleimputer

In [6]:
nan_counts = x_train.isna().sum()
print(nan_counts[nan_counts > 0])

year           13
cylinders    2312
dtype: int64


In [7]:
print(x_train.columns.tolist())

['id', 'region', 'year', 'manufacturer', 'model', 'condition', 'cylinders', 'fuel', 'odometer', 'title_status', 'transmission', 'drive', 'type', 'log_price', 'model_clean']


In [8]:
print("--- Linear Regression ---")
model = Pipeline([
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()),
    ('lin_reg', LinearRegression())
])
evaluate_model(model, x_train, y_train, x_val, y_val)

--- Linear Regression ---
Cross-Validation R2: 0.5923, Training CV RMSE: 9473.4021
Training R2: 0.6235, Training RMSE: 9113.3462
Validation R2: 0.5315, Validation RMSE: 10506.8040



siden training verdiene og validation er veldig nærme hverandre, indikerer det ingen over eller underfitting

# Decision Tree Regressor

In [9]:
print("--- Decision Tree Regressor ---")
model = Pipeline([
    ('preprocessor', preprocessor),
    ('DecisionTreeRegressor', DecisionTreeRegressor(random_state=42))
])
evaluate_model(model, x_train, y_train, x_val, y_val)

--- Decision Tree Regressor ---
Cross-Validation R2: 0.4541, Training CV RMSE: 10951.7603
Training R2: 1.0000, Training RMSE: 14.3315
Validation R2: 0.4399, Validation RMSE: 11488.9184



Denne modellen er overfittet. modellen har memorisert dataen

# Random Forest Regressor

In [10]:
print("--- Random Forest Regressor ---")
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(max_depth=10, random_state=42))
])
evaluate_model(model, x_train, y_train, x_val, y_val)

--- Random Forest Regressor ---
Cross-Validation R2: 0.7055, Training CV RMSE: 8051.9941
Training R2: 0.8773, Training RMSE: 5202.0727
Validation R2: 0.6790, Validation RMSE: 8696.8277



### Overfitted

# XGBoost Regressor

In [11]:
print("--- XGBoost Regressor ---")
model = Pipeline([
    ('preprocessor', preprocessor),
    ('xgb', XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror',
        random_state=42
    ))
])
evaluate_model(model, x_train, y_train, x_val, y_val)

--- XGBoost Regressor ---
Cross-Validation R2: 0.7434, Training CV RMSE: 7508.8510
Training R2: 0.8940, Training RMSE: 4836.1536
Validation R2: 0.7317, Validation RMSE: 7951.9374



In [20]:
print("--- XGBoost Regressor ---")
model = Pipeline([
    ('preprocessor', preprocessor),
    ('xgb', XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror',
        random_state=42,
        reg_alpha=1.0,
        reg_lambda=1.0
    ))
])
evaluate_model(model, x_train, y_train, x_val, y_val)

--- XGBoost Regressor ---
Cross-Validation R2: 0.7394, Training CV RMSE: 7571.4516
Training R2: 0.8723, Training RMSE: 5307.8872
Validation R2: 0.7238, Validation RMSE: 8067.8438



In [21]:
pred = model.predict(x_val)
ape = (abs(y_val - pred) / y_val)
print("MAE:", abs(y_val - pred).mean(), "Median APE:", ape.median())

MAE: 4855.520454986941 Median APE: 0.19971973824302283


In [22]:
y_train.describe()

count      5624.000000
mean      19269.014936
std       14854.097673
min         515.000000
25%        7900.000000
50%       15722.000000
75%       27980.000000
max      145000.000000
Name: price, dtype: float64

In [23]:
x_train.assign(price=y_train).nlargest(10, 'price')

,id,region,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,drive,type,log_price,model_clean,price
7628,7302901816,anchorage / mat-su,2014,unknown,freightliner m2 106,like new,6.0,diesel,17094.0,clean,automatic,4wd,truck,11.884496,other,145000
44110,7316014285,palm springs,1959,chevrolet,corvette,like new,8.0,gas,47000.0,clean,manual,unknown,convertible,11.736077,corvette,125000
253221,7316847330,north jersey,2017,audi,unknown,unknown,10.0,other,16613.0,clean,automatic,fwd,other,11.715221,unknown,122420
417324,7313557179,green bay,2017,audi,unknown,unknown,10.0,other,16613.0,clean,automatic,fwd,other,11.715221,unknown,122420
370841,7315196841,dallas / fort worth,1965,chevrolet,corvette sting ray,unknown,NaN,gas,462.0,clean,manual,unknown,convertible,11.608063,other,109980
278718,7313411296,rochester,2013,dodge,srt viper,unknown,NaN,gas,3600.0,clean,manual,unknown,unknown,11.599112,other,109000
4826,7316436488,tuscaloosa,2020,bmw,m805i,excellent,8.0,gas,9000.0,clean,automatic,4wd,sedan,11.560772,other,104900
44389,7314331344,palm springs,1997,ferrari,f355 spider,excellent,8.0,gas,22000.0,clean,manual,unknown,convertible,11.512925,other,99999
203325,7307707091,holland,1970,pontiac,gto judge,excellent,8.0,gas,782.0,clean,manual,rwd,unknown,11.492733,other,98000
384322,7316790438,waco,2018,dodge,unknown,unknown,8.0,other,3591.0,clean,automatic,fwd,other,11.422465,unknown,91350


In [28]:
cols_to_check = [col for col in df.columns if col not in ['id', 'region']]
antall_duplikater = df.duplicated(subset=cols_to_check).sum()
print(antall_duplikater)

NameError: name 'df' is not defined

# CatBoost

In [ ]:
# --- CatBoost Regressor ---
print("--- CatBoost Regressor ---")
model_catboost = Pipeline([
    ('preprocessor', preprocessor),
    ('catboost', CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        verbose=0,
        random_seed=42
    ))
])

evaluate_model(model_catboost, x_train, y_train, x_val, y_val)

# AdaBoost

In [ ]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor

# --- AdaBoost Regressor ---
print("--- AdaBoost Regressor ---")
model_adaboost = Pipeline([
    ('preprocessor', preprocessor),
    ('adaboost', AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=4),
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    ))
])

evaluate_model(model_adaboost, x_train, y_train, x_val, y_val)

In [24]:
from sklearn.compose import TransformedTargetRegressor

base = Pipeline([
    ('preprocessor', preprocessor),
    ('xgb', XGBRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=1.0, reg_lambda=1.0, random_state=42
    ))
])

model = TransformedTargetRegressor(
    regressor=base, func=np.log1p, inverse_func=np.expm1
)
evaluate_model(model, x_train, y_train, x_val, y_val)

Cross-Validation R2: 0.7100, Training CV RMSE: 7991.9729
Training R2: 0.8166, Training RMSE: 6361.5633
Validation R2: 0.6962, Validation RMSE: 8461.8121

